# 17 Seizure vs rest: class imbalance and the cut-off

Seizure is 20% of segments. In 09 the ranking was near perfect (AUC 0.997) but about 11 of 80 seizure segments were missed at the 0.5 cut-off. Do imbalance fixes or a lower cut-off help? Chunk level, twin-grouped folds, development data only, 5 repeats. Resampling is done inside each training fold (imblearn pipeline), never on validation data.

In [1]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import make_pipeline
from imblearn.over_sampling import RandomOverSampler, SMOTE

from src.config import PROCESSED, RANDOM_STATE
from src.cv import load_fold_table, fold_of_rows
from src.data import feature_cols
from src.evaluate import cross_validate, out_of_fold_proba, score

dev = pd.read_csv(PROCESSED / "dev_allclass.csv")
table = load_fold_table("cv_folds_allclass.csv")
X = dev[feature_cols(dev)].to_numpy(dtype=float)
log_amplitude = np.log10(X.std(axis=1, keepdims=True))
three = np.c_[log_amplitude[:, 0], stats.kurtosis(X, axis=1), np.abs(np.diff(X, axis=1)).mean(axis=1)]
target = (dev["y"] == 1).to_numpy()

fixes = {
    "no adjustment": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)),
    "class_weight balanced": lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight="balanced")),
    "random oversampling": lambda: make_pipeline(StandardScaler(), RandomOverSampler(random_state=RANDOM_STATE), LogisticRegression(max_iter=1000)),
    "SMOTE": lambda: make_pipeline(StandardScaler(), SMOTE(random_state=RANDOM_STATE), LogisticRegression(max_iter=1000)),
}
print("chunk rows:", len(dev), "| seizure share of chunks:", round(target.mean(), 2))

chunk rows: 9200 | seizure share of chunks: 0.2


## Fixes at the 0.5 cut-off

In [2]:
rows = {}
for fset, F in {"log amplitude": log_amplitude, "amplitude, kurtosis, line length": three}.items():
    for name, mk in fixes.items():
        rows[(fset, name)] = cross_validate(mk, dev, F, table, positive=1).drop(columns="repeat").mean()
res = pd.DataFrame(rows).T
res[["auc_fold", "pr_auc_fold", "sensitivity", "specificity", "precision", "balanced_accuracy", "accuracy", "fn", "fp"]].round(3)

auc_fold  pr_auc_fold  \
log amplitude                    no adjustment             0.997        0.991   
                                 class_weight balanced     0.998        0.992   
                                 random oversampling       0.998        0.992   
                                 SMOTE                     0.998        0.992   
amplitude, kurtosis, line length no adjustment             0.996        0.988   
                                 class_weight balanced     0.997        0.991   
                                 random oversampling       0.997        0.990   
                                 SMOTE                     0.997        0.990   

                                                        sensitivity  \
log amplitude                    no adjustment                0.865   
                                 class_weight balanced        0.975   
                                 random oversampling          0.975   
                                 SMOTE                        0.975   
amplitude, kurtosis, line length no adjustment                0.838   
                                 class_weight balanced        0.975   
                                 random oversampling          0.975   
                                 SMOTE                        0.975   

                                                        specificity  \
log amplitude                    no adjustment                0.991   
                                 class_weight balanced        0.976   
                                 random oversampling          0.976   
                                 SMOTE                        0.976   
amplitude, kurtosis, line length no adjustment                0.991   
                                 class_weight balanced        0.961   
                                 random oversampling          0.962   
                                 SMOTE                        0.962   

                                                        precision  \
log amplitude                    no adjustment              0.958   
                                 class_weight balanced      0.909   
                                 random oversampling        0.911   
                                 SMOTE                      0.911   
amplitude, kurtosis, line length no adjustment              0.960   
                                 class_weight balanced      0.863   
                                 random oversampling        0.865   
                                 SMOTE                      0.867   

                                                        balanced_accuracy  \
log amplitude                    no adjustment                      0.928   
                                 class_weight balanced              0.975   
                                 random oversampling                0.976   
                                 SMOTE                              0.976   
amplitude, kurtosis, line length no adjustment                      0.914   
                                 class_weight balanced              0.968   
                                 random oversampling                0.968   
                                 SMOTE                              0.969   

                                                        accuracy    fn    fp  
log amplitude                    no adjustment             0.965  10.8   3.0  
                                 class_weight balanced     0.976   2.0   7.8  
                                 random oversampling       0.976   2.0   7.6  
                                 SMOTE                     0.976   2.0   7.6  
amplitude, kurtosis, line length no adjustment             0.960  13.0   2.8  
                                 class_weight balanced     0.964   2.0  12.4  
                                 random oversampling       0.964   2.0  12.2  
                                 SMOTE                     0.965   2.0  12.0

## Cut-off sweep

Repeat 0 only, no adjustment, log amplitude. One out-of-fold probability per segment. This is descriptive: the cut-off is read off the same predictions it is scored on, so it is not a tuned or validated choice.

In [3]:
fold0 = fold_of_rows(dev, table, 0)
proba = out_of_fold_proba(fixes["no adjustment"], log_amplitude, target, fold0)
sweep = {t: score(dev, target, proba, fold0, threshold=t) for t in (0.5, 0.4, 0.3, 0.2, 0.1)}
pd.DataFrame(sweep).T[["sensitivity", "specificity", "precision", "fn", "fp"]].round(3)

,sensitivity,specificity,precision,fn,fp
0.5,0.875,0.991,0.959,10.0,3.0
0.4,0.925,0.988,0.949,6.0,4.0
0.3,0.950,0.978,0.916,4.0,7.0
0.2,0.988,0.944,0.814,1.0,18.0
0.1,1.000,0.922,0.762,0.0,25.0


## The missed seizures at 0.5

In [4]:
seg = (pd.DataFrame({"segment": dev["segment"], "target": target, "p": proba, "amp": X.std(axis=1)})
       .groupby("segment").agg(target=("target", "first"), p=("p", "mean"), amp=("amp", "median")))
missed = seg[seg["target"] & (seg["p"] <= 0.5)]
caught = seg[seg["target"] & (seg["p"] > 0.5)]
rest = seg[~seg["target"]]
print(f"seizure segments missed at 0.5: {len(missed)} of {seg['target'].sum()}")
print(f"median chunk amplitude (SD of the 178 samples, median over chunks): missed {missed['amp'].median():.0f} | caught {caught['amp'].median():.0f} | non-seizure {rest['amp'].median():.0f}")
print("missed amplitudes:", np.sort(missed["amp"].round(0).to_numpy()))
print("non-seizure segments above the lowest missed amplitude:", int((rest["amp"] > missed["amp"].min()).sum()), "of", len(rest))

seizure segments missed at 0.5: 10 of 80
median chunk amplitude (SD of the 178 samples, median over chunks): missed 112 | caught 294 | non-seizure 45
missed amplitudes: [ 94.  98. 102. 103. 110. 114. 121. 123. 125. 126.]
non-seizure segments above the lowest missed amplitude: 12 of 320


## Findings

- **The ranking was never the problem.** AUC is 0.997 with no adjustment and 0.998 with any fix. Differences this small are ties.
- **The fixes only move the cut-off.** Class weights, oversampling and SMOTE all give the same result for log amplitude: sensitivity 0.865 to 0.975, specificity 0.991 to 0.976. Missed seizures fall from about 10.8 to 2.0 of 80, false alarms rise from 3.0 to about 7.6 of 320. SMOTE is no better than plain duplication here (one feature, so there is little to interpolate).
- **The three-number set pays more for it:** sensitivity 0.975 but specificity 0.962 (about 12 false alarms).
- **Cut-off sweep (repeat 0, descriptive only):** 0.5 gives 0.875 / 0.991; 0.3 gives 0.950 / 0.978; 0.1 gives 1.000 / 0.922. Lowering the cut-off is the same trade as the fixes, with the choice made by hand.
- **Missed seizures:** 10 of 80 at 0.5. Their median chunk amplitude is 112, against 294 for caught seizures and 45 for non-seizure. They are small seizures, in the overlap with the loudest non-seizure segments (12 of 320 non-seizure segments are above the lowest missed one).

## Caveats

- Chunk-level, one feature, logistic regression only. Development data only.
- The sweep reads cut-offs off the same predictions it scores. It shows the trade, it does not pick a validated cut-off.
- Which trade is right depends on the cost of a missed seizure against a false alarm. That is a clinical question and not answered here.